# PSMPY MODULE - please change data I am using Dummy data

In [50]:
import numpy as np
import pandas as pd
from psmpy import PsmPy
from scipy.spatial.distance import cdist

rng = np.random.default_rng(42)

# ======================= SETTINGS =======================
N_TEST = 20
N_POOL = 180
K = 3                        # controls wanted per test store
MATCH_ON = "mahalanobis"     # "mahalanobis" (features directly) or "propensity" (PsmPy score)
CALIPER_Q = 0.05             # mahalanobis: keep pairs in the closest 5% of all test-control distances
CALIPER_SD = 0.3             # propensity: max logit gap = 0.3 x SD of the logit score
TRUE_LIFT = 0.06             # only used by the dummy data

ID_COL = "store_id"
TREAT_COL = "is_test"

# Matching features: pre-test columns ONLY. No outcome columns, no region code.
features = ["avg_weekly_sales", "sq_ft", "avg_basket",
            "median_income", "competitors_5mi", "pre_sales_trend"]

# ======================= 1. DUMMY DATA =======================
# Replace this block with: df = pd.read_csv("your_stores.csv")
n = N_TEST + N_POOL
is_test = np.r_[np.ones(N_TEST), np.zeros(N_POOL)].astype(int)
rng.shuffle(is_test)

sq_ft = rng.normal(75000, 15000, n).clip(35000, 120000)
income = rng.normal(70000, 18000, n).clip(30000, 150000)
sales = (sq_ft * 9 + income * 1.5 + rng.normal(0, 40000, n)).clip(300000, None)
sq_ft[is_test == 1] += 8000
income[is_test == 1] += 7000
sales[is_test == 1] += 60000

df = pd.DataFrame({
    "store_id": [f"HEB_{i:04d}" for i in range(n)],
    "region": rng.integers(1, 41, n),            # kept in output, NOT used for matching
    "is_test": is_test,
    "avg_weekly_sales": sales,
    "sq_ft": sq_ft,
    "avg_basket": rng.normal(62, 9, n).clip(35, 95),
    "median_income": income,
    "competitors_5mi": rng.poisson(3, n),
    "pre_sales_trend": rng.normal(0.01, 0.02, n),
})
df["sales_pre"] = df["avg_weekly_sales"]
df["sales_post"] = (df["sales_pre"]
                    * (1 + df["pre_sales_trend"] * 8 + rng.normal(0, 0.015, n))
                    * (1 + TRUE_LIFT * df["is_test"]))

# ======================= 2. PsmPy PROPENSITY SCORES =======================
# Give PsmPy ONLY: ID + treatment + matching features. Outcomes cannot leak in.
psm_df = df[[ID_COL, TREAT_COL] + features].copy()

psm = PsmPy(psm_df, treatment=TREAT_COL, indx=ID_COL)
psm.logistic_ps(balance=True)
scores = psm.predicted_data.copy()

if ID_COL not in scores.columns:
    scores = scores.reset_index().rename(columns={"index": ID_COL})
if "propensity_logit" not in scores.columns:
    p = scores["propensity_score"].clip(1e-6, 1 - 1e-6)
    scores["propensity_logit"] = np.log(p / (1 - p))

# safety check: stop if outcome or label columns leaked into the model
assert not any(c in scores.columns for c in ["sales_pre", "sales_post", "region"]), \
    "Outcome/label columns leaked into the propensity model"
print("Columns from PsmPy:", scores.columns.tolist())

scores = scores.set_index(ID_COL)

# full table (all columns) + propensity scores
d = df.set_index(ID_COL)
d["propensity_score"] = scores.loc[d.index, "propensity_score"]
d["propensity_logit"] = scores.loc[d.index, "propensity_logit"]

test_ids = d.index[d[TREAT_COL] == 1]
ctrl_ids = d.index[d[TREAT_COL] == 0]

# ======================= 3. DISTANCE MATRIX (test x control) =======================
if MATCH_ON == "mahalanobis":
    X = d[features].values.astype(float)
    VI = np.linalg.pinv(np.cov(X, rowvar=False))          # handles correlated columns
    D = pd.DataFrame(cdist(d.loc[test_ids, features].values.astype(float),
                           d.loc[ctrl_ids, features].values.astype(float),
                           metric="mahalanobis", VI=VI),
                     index=test_ids, columns=ctrl_ids)
    cutoff = np.quantile(D.values, CALIPER_Q)
else:
    tl = d.loc[test_ids, "propensity_logit"].values
    cl = d.loc[ctrl_ids, "propensity_logit"].values
    D = pd.DataFrame(np.abs(tl[:, None] - cl[None, :]), index=test_ids, columns=ctrl_ids)
    cutoff = CALIPER_SD * d["propensity_logit"].std()

print(f"\nMatch method: {MATCH_ON} | caliper cutoff: {cutoff:.4f}")

# ======================= 4. MATCHING: K nearest, caliper, controls reusable =======================
rows = []
dropped = []
for t_id in D.index:
    best = D.loc[t_id].sort_values().head(K)      # K nearest controls (pool is never shrunk)
    best = best[best <= cutoff]                   # caliper: poor matches dropped
    if len(best) == 0:
        dropped.append(t_id)
    for rank, (c_id, dist) in enumerate(best.items(), start=1):
        rows.append({"test_store": t_id, "control_store": c_id,
                     "rank": rank, "distance": dist})

pairs = pd.DataFrame(rows, columns=["test_store", "control_store", "rank", "distance"])

print("Test stores matched:", pairs.test_store.nunique(), "/", N_TEST)
print("Dropped test stores:", dropped)
print("Controls per test store (count of test stores):")
print(pairs.groupby("test_store").size().value_counts().sort_index())
print("Unique control stores used:", pairs.control_store.nunique())
print("\nMost reused control stores:")
print(pairs.control_store.value_counts().head(10))

# ======================= 5. BALANCE CHECK (SMD) =======================
test_all = d.loc[test_ids]
ctrl_all = d.loc[ctrl_ids]
test_m = d.loc[pairs.test_store.unique()]
ctrl_m = d.loc[pairs.control_store]               # repeats allowed (reuse)

print("\nBalance (want |SMD| < 0.1 after matching):")
for f in features:
    a, b = test_all[f], ctrl_all[f]
    before = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    a, b = test_m[f], ctrl_m[f]
    after = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    print(f"{f:20s} before={before:6.3f}   after={after:6.3f}")

# ======================= 6. LIFT (difference-in-differences) =======================
lifts = []
for t_id, g in pairs.groupby("test_store"):
    t = d.loc[t_id]
    c = d.loc[g.control_store]
    lifts.append((t.sales_post / t.sales_pre - 1)
                 - (c.sales_post / c.sales_pre - 1).mean())
lifts = np.array(lifts)

print(f"\nEstimated lift: {lifts.mean():.2%}   (true simulated = {TRUE_LIFT:.0%})")
print(f"Std error: {lifts.std(ddof=1) / np.sqrt(len(lifts)):.2%}")

# ======================= 7. OUTPUT WITH ALL COLUMNS =======================
show_cols = [c for c in d.columns if c != TREAT_COL]
test_part = d.loc[pairs.test_store, show_cols].add_prefix("test_").reset_index(drop=True)
ctrl_part = d.loc[pairs.control_store, show_cols].add_prefix("ctrl_").reset_index(drop=True)
full_pairs = pd.concat([pairs.reset_index(drop=True), test_part, ctrl_part], axis=1)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
print("\nMatched pairs with all columns (first 6 rows):")
print(full_pairs.head(6).round(3).to_string())

full_pairs.to_csv("Edith_matched_pairs_all_columns.csv", index=False)
print("\nSaved: edtih_matched_pairs_all_columns.csv")

Columns from PsmPy: ['store_id', 'avg_weekly_sales', 'sq_ft', 'avg_basket', 'median_income', 'competitors_5mi', 'pre_sales_trend', 'propensity_score', 'propensity_logit', 'is_test']

Match method: mahalanobis | caliper cutoff: 1.9283
Test stores matched: 20 / 20
Dropped test stores: []
Controls per test store (count of test stores):
1     1
2     1
3    18
Name: count, dtype: int64
Unique control stores used: 47

Most reused control stores:
control_store
HEB_0156    2
HEB_0177    2
HEB_0008    2
HEB_0036    2
HEB_0078    2
HEB_0077    2
HEB_0081    2
HEB_0086    2
HEB_0168    2
HEB_0179    2
Name: count, dtype: int64

Balance (want |SMD| < 0.1 after matching):
avg_weekly_sales     before= 0.611   after= 0.221
sq_ft                before= 0.695   after= 0.277
avg_basket           before= 0.242   after=-0.023
median_income        before= 0.425   after= 0.046
competitors_5mi      before= 0.111   after=-0.001
pre_sales_trend      before=-0.250   after=-0.060

Estimated lift: 5.65%   (true 

## Method 2 need refinement

In [33]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(42)

# ---------------- SETTINGS ----------------
N_TEST = 20
N_POOL = 180
K = 3               # controls wanted per test store
CALIPER_SD = 0.2   # caliper = this * SD of logit(ps). Bigger = more matches, lower quality
TREND_TOL = 0.01    # optional hard rule on pre-trend. Set to None to turn off
TRUE_LIFT = 0.06
features = ["avg_weekly_sales", "sq_ft", "avg_basket",
            "median_income", "competitors_5mi", "pre_sales_trend"]

# ---------------- 1. DUMMY DATA ----------------
n = N_TEST + N_POOL
is_test = np.r_[np.ones(N_TEST), np.zeros(N_POOL)].astype(int)
rng.shuffle(is_test)

sq_ft = rng.normal(75000, 15000, n).clip(35000, 120000)
income = rng.normal(70000, 18000, n).clip(30000, 150000)
sales = (sq_ft * 9 + income * 1.5 + rng.normal(0, 40000, n)).clip(300000, None)
sq_ft[is_test == 1] += 8000          # make test stores a bit bigger/richer
income[is_test == 1] += 7000
sales[is_test == 1] += 60000

df = pd.DataFrame({
    "store_id": [f"HEB_{i:04d}" for i in range(n)],
    "region": rng.integers(1, 41, n),          # region 1..40, NOT used in matching
    "is_test": is_test,
    "avg_weekly_sales": sales,
    "sq_ft": sq_ft,
    "avg_basket": rng.normal(62, 9, n).clip(35, 95),
    "median_income": income,
    "competitors_5mi": rng.poisson(3, n),
    "pre_sales_trend": rng.normal(0.01, 0.02, n),
})
df["sales_pre"] = df["avg_weekly_sales"]
df["sales_post"] = (df["sales_pre"]
                    * (1 + df["pre_sales_trend"] * 8 + rng.normal(0, 0.015, n))
                    * (1 + TRUE_LIFT * df["is_test"]))

# ---------------- 2. PROPENSITY SCORE ----------------
X = StandardScaler().fit_transform(df[features])
lr = LogisticRegression(C=1.0, max_iter=1000).fit(X, df["is_test"])
df["pscore"] = lr.predict_proba(X)[:, 1]
df["logit_ps"] = np.log(df["pscore"] / (1 - df["pscore"]))

# ---------------- 3. MATCHING WITH CALIPER ----------------
caliper = CALIPER_SD * df["logit_ps"].std()
test = df[df.is_test == 1]
ctrl = df[df.is_test == 0]

rows = []
dropped = []
for _, t in test.iterrows():
    pool = ctrl
    if TREND_TOL is not None:
        pool = pool[(pool.pre_sales_trend - t.pre_sales_trend).abs() <= TREND_TOL]
    if len(pool) == 0:
        dropped.append(t.store_id)
        continue

    nn = NearestNeighbors(n_neighbors=min(K, len(pool))).fit(pool[["logit_ps"]].values)
    dist, idx = nn.kneighbors([[t.logit_ps]])

    kept = 0
    for d, i in zip(dist[0], idx[0]):
        if d <= caliper:                      # poor matches are dropped here
            rows.append({"test_store": t.store_id,
                         "control_store": pool.iloc[i].store_id,
                         "ps_distance": d})
            kept += 1
    if kept == 0:
        dropped.append(t.store_id)

pairs = pd.DataFrame(rows, columns=["test_store", "control_store", "ps_distance"])

print("Caliper:", round(caliper, 4))
print("Test stores matched:", pairs.test_store.nunique(), "/", N_TEST)
print("Dropped test stores:", dropped)
print("Controls per test store:")
print(pairs.groupby("test_store").size().value_counts().sort_index())
print("Unique control stores used:", pairs.control_store.nunique())

# ---------------- 4. BALANCE CHECK (SMD) ----------------
d = df.set_index("store_id")
test_m = d.loc[pairs.test_store.unique()]
ctrl_m = d.loc[pairs.control_store]            # repeats allowed (reuse)

print("\nBalance (want |SMD| < 0.1 after):")
for f in features:
    a, b = test[f], ctrl[f]
    before = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    a, b = test_m[f], ctrl_m[f]
    after = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    print(f"{f:20s} before={before:6.3f}  after={after:6.3f}")

# ---------------- 5. LIFT (difference-in-differences) ----------------
lifts = []
for t_id, g in pairs.groupby("test_store"):
    t = d.loc[t_id]
    c = d.loc[g.control_store]
    t_growth = t.sales_post / t.sales_pre - 1
    c_growth = (c.sales_post / c.sales_pre - 1).mean()
    lifts.append(t_growth - c_growth)

lifts = np.array(lifts)
print(f"\nEstimated lift: {lifts.mean():.2%}  (true = {TRUE_LIFT:.0%})")
print(f"Std error: {lifts.std(ddof=1) / np.sqrt(len(lifts)):.2%}")

# Final list: each test store with its controls
print("\nMatched pairs:")
print(pairs.groupby("test_store")["control_store"].apply(list))

Caliper: 0.2039
Test stores matched: 18 / 20
Dropped test stores: ['HEB_0084', 'HEB_0087']
Controls per test store:
1     3
2     3
3    12
Name: count, dtype: int64
Unique control stores used: 37

Balance (want |SMD| < 0.1 after):
avg_weekly_sales     before= 0.611  after=-0.210
sq_ft                before= 0.695  after=-0.090
avg_basket           before= 0.242  after= 0.180
median_income        before= 0.425  after= 0.072
competitors_5mi      before= 0.111  after= 0.299
pre_sales_trend      before=-0.250  after=-0.046

Estimated lift: 7.40%  (true = 6%)
Std error: 0.75%

Matched pairs:
test_store
HEB_0005    [HEB_0168, HEB_0139, HEB_0077]
HEB_0034    [HEB_0037, HEB_0172, HEB_0033]
HEB_0042              [HEB_0168, HEB_0139]
HEB_0047    [HEB_0010, HEB_0126, HEB_0175]
HEB_0058    [HEB_0076, HEB_0018, HEB_0184]
HEB_0061              [HEB_0032, HEB_0018]
HEB_0071    [HEB_0044, HEB_0128, HEB_0126]
HEB_0074    [HEB_0117, HEB_0062, HEB_0153]
HEB_0105                        [HEB_0185]
HEB_013